# CREACIÓN DE RED NEURONAL PARA GENERAR EMBEDDINGS CON TENSORFLOW

# paso 1 - importamos librerias

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# paso 2 - Dataset de ejemplo

In [2]:
sentences = [
    "I love deep learning",
    "I love machine learning",
    "deep learning is fun"
]

# paso 3 - Tokenizar las frases (convertir las palabras a índices)

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [4]:
tokenizer

# paso 4 - creamos el vocabulario

In [5]:
vocab_size = len(tokenizer.word_index) + 1  # Añadimos 1 para considerar el índice 0 (pad)
print("Vocabulary size:", vocab_size)
tokenizer.word_index

Vocabulary size: 8


{'learning': 1, 'i': 2, 'love': 3, 'deep': 4, 'machine': 5, 'is': 6, 'fun': 7}

# paso 5 - Convertir las frases a secuencias de índices

In [6]:
sequences = tokenizer.texts_to_sequences(sentences)

In [7]:
sequences

[[2, 3, 4, 1], [2, 3, 5, 1], [4, 1, 6, 7]]

# Pad sequences para asegurar que todas las secuencias tengan la misma longitud

In [8]:
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print("Padded sequences:", padded_sequences)

Padded sequences: [[2 3 4 1]
 [2 3 5 1]
 [4 1 6 7]]


# CREAMOS EL MODELO DE RED NEURONAL PARA EMBEDDING

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

In [10]:
# Definir el modelo
embedding_dim = 8  # Dimensión de los embeddings (cada palabra será representada por un vector de 8 números)

model = Sequential()

# Capa de Embeddings
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length))

# Capa de GlobalAveragePooling1D para hacer un resumen de la secuencia
model.add(GlobalAveragePooling1D())

# Capa densa para clasificación (en este caso, un ejemplo de regresión)
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# COMPILAMOS EL MODELO DE RED NEURONAL

In [11]:
# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Resumen del modelo
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# ENTRENAMOS EL MODELO

In [12]:
# Etiquetas de ejemplo para clasificación binaria (por ejemplo, 0 o 1)
labels = [1, 0, 1]  # Aquí se podría tener una etiqueta binaria para cada frase

# Convertir las etiquetas en un tensor
labels_tensor = tf.convert_to_tensor(labels)

# Entrenar el modelo
model.fit(padded_sequences, labels_tensor, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.6904   
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.6893
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.6881
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.6869
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6667 - loss: 0.6857


In [13]:
# Obtener los embeddings aprendidos por la capa
embeddings = model.layers[0].get_weights()[0]
print("Embeddings learned by the model:")
print(embeddings)

Embeddings learned by the model:
[[ 0.03011048  0.00139835 -0.00464303  0.03590963 -0.03089894 -0.01973013
  -0.03420601  0.03902263]
 [-0.0361742   0.05156029 -0.01492917 -0.01183701 -0.03445967  0.00458749
   0.00168201 -0.0285701 ]
 [ 0.03759412 -0.02961086  0.03822311  0.01483675  0.03492058  0.04557027
   0.01589581  0.01149185]
 [ 0.03478176 -0.01956152 -0.02287397 -0.0065963  -0.01970831  0.04573822
   0.00096717  0.0465116 ]
 [ 0.04934907 -0.04003676  0.04967695  0.04030346  0.02162569 -0.01162431
  -0.0201695  -0.04487554]
 [-0.00986032 -0.03211308  0.00542793  0.01084841  0.0470505  -0.02218863
   0.01858967 -0.04416391]
 [ 0.00916599  0.03096541  0.00367642  0.00188386 -0.02934302 -0.02757087
  -0.05077576  0.03813509]
 [ 0.04906829 -0.04034938  0.01021319  0.02480062  0.04437913 -0.03008662
   0.00842834 -0.00763766]]
